In [1]:
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Union, Optional
import draccus
from accelerate.utils import set_seed
from vlm_eval.conf import DatasetConfig, DatasetRegistry
from vlm_eval.models import load_vlm
from vlm_eval.overwatch import initialize_overwatch
from vlm_eval.tasks import get_task_runner
from prismatic.models import get_llm_backbone_and_tokenizer, get_vision_backbone_and_transform, get_vlm
from prismatic.preprocessing import get_dataset_and_collator
from prismatic.util.batching_utils import SplitModalitySampler
from torch.utils.data import DistributedSampler, DataLoader
import torch

# Configuration
RUN_DIR = '/home/aac/ssrivas9/prismatic-vlms/runs/reproduction-align-pythia+410m-test_mi300x'
hf_token = None
device = "cuda"
batch_size = 16

# Load the VLM
vlm = load_vlm('prismatic', 'prism-clip+7b', RUN_DIR, hf_token=hf_token, device=device)

# Change directory
os.chdir('/home/aac/ssrivas9/prismatic-vlms')
tokenizer, image_transform = vlm.model.llm_backbone.tokenizer, vlm.model.vision_backbone.image_transform

# Loop over stages to save one batch each
for stage in ["align", "finetune"]:
    # Dataset and collator setup
    dataset_cfg: DatasetConfig = DatasetConfig.get_choice_class('llava-v15')
    train_dataset, collator = get_dataset_and_collator(
        stage,
        dataset_cfg,
        image_transform,
        tokenizer,
        prompt_builder_fn=vlm.model.llm_backbone.prompt_builder_fn,
        default_image_resolution=vlm.model.vision_backbone.default_image_resolution,
        padding_side=tokenizer.padding_side,
    )

    # Sampler setup
    if stage == "align":
        sampler = DistributedSampler(
            train_dataset,
            num_replicas=1,
            rank=0,
            shuffle=True,
            seed=123,
            drop_last=False,
        )
    else:
        modality_lengths = train_dataset.get_modality_lengths()
        sampler = SplitModalitySampler(
            train_dataset,
            modality_lengths,
            global_batch_size=batch_size,  # Corrected to match DataLoader batch_size
            num_replicas=1,
            rank=0,
            seed=123,
            drop_last=False,
        )

    # DataLoader setup
    dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=sampler,
        collate_fn=collator,
        num_workers=12,
        worker_init_fn=None,
    )

    # Get and save the first batch
    vlm.model.train()
    with torch.autocast("cuda", dtype=torch.float32):
        for batch_idx, batch in enumerate(dataloader):
            # Move batch to device
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            
            # Move tensors to CPU and save
            cpu_batch = {k: v.cpu() if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            save_path = f'batch_{stage}.pt'
            torch.save(cpu_batch, save_path)
            print(f"Saved batch for stage {stage} to {save_path}")
            
            # Stop after the first batch
            break

/opt/conda/envs/eval/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loading prismatic <vlm_eval.models.prismatic.PrismaticVLM object at 0x7002ec994250>


03/06 [19:18:37] INFO     | >> [*] Loading from local path                                               ]8;id=293902;file:///opt/conda/envs/eval/lib/python3.10/site-packages/prismatic/models/load.py\load.py]8;;\:]8;id=213165;file:///opt/conda/envs/eval/lib/python3.10/site-packages/prismatic/models/load.py#56\56]8;;\
                          `/home/aac/ssrivas9/prismatic-vlms/runs/reproduction-align-pythia+410m-test_mi           
                          300x`                                                                                    

                 INFO     | >> [*]                                                                       ]8;id=537370;file:///opt/conda/envs/eval/lib/python3.10/site-packages/prismatic/models/load.py\load.py]8;;\:]8;id=94628;file:///opt/conda/envs/eval/lib/python3.10/site-packages/prismatic/models/load.py#71\71]8;;\
                                                                                                                   
                          🚀 (LATEST) Loading Prismatic VLM for Inference 🚀 🚀                                    
                                                                                                                   
                                                                                                                   

                 INFO     | >> [*] Found Config =>> Loading & Freezing stage0-pythia+410m with:          ]8;id=569343;file:///opt/conda/envs/eval/lib/python3.10/site-packages/prismatic/models/load.py\load.py]8;;\:]8;id=73422;file:///opt/conda/envs/eval/lib/python3.10/site-packages/prismatic/models/load.py#79\79]8;;\
                                       Vision Backbone =>> clip-vit-l-336px                                        
                                       LLM Backbone    =>> pythia-410m                                             
                                       Arch Specifier  =>> gelu-mlp                                                
                                       Mitigation Strategy      =>> None             LoRA      =>> rank:           
                          16, alpha: 8, lora_target_modules: all-linear             Load in 8bit     =>>           
                          False             Checkpoint Path =>>                                                    
                          `/home/aac/ssrivas9/prismatic-vlms/runs/reproduction-align-pythia+410m-test_mi           
                          300x/checkpoints/latest-checkpoint.pt`                                                   

...

03/06 [19:18:42] INFO     | >> [*] Dimensions of Projector:                                         ]8;id=187039;file:///opt/conda/envs/eval/lib/python3.10/site-packages/prismatic/models/vlms/prismatic.py\prismatic.py]8;;\:]8;id=715305;file:///opt/conda/envs/eval/lib/python3.10/site-packages/prismatic/models/vlms/prismatic.py#85\85]8;;\
                           vision_dim: 1024, llm_dim: 1024                                                         

                 INFO     | >> [*] Total Parameters: 2097152                                        ]8;id=834105;file:///opt/conda/envs/eval/lib/python3.10/site-packages/prismatic/models/vlms/prismatic.py\prismatic.py]8;;\:]8;id=700553;file:///opt/conda/envs/eval/lib/python3.10/site-packages/prismatic/models/vlms/prismatic.py#86\86]8;;\

                 ERROR    | >>     |=> Loading only `projector` from checkpoint                    ]8;id=518704;file:///opt/conda/envs/eval/lib/python3.10/site-packages/prismatic/models/vlms/prismatic.py\prismatic.py]8;;\:]8;id=421926;file:///opt/conda/envs/eval/lib/python3.10/site-packages/prismatic/models/vlms/prismatic.py#152\152]8;;\

loaded prismatic prism-clip+7b


KeyError: 'llava-v15'